# RQ3 / E5 — запуск в Colab

CE vs SupCon vs CE + SupCon для одного E5 (`intfloat/e5-base-v2`), общий linear probe,
оценка на hold-out AEGIS и OOD ToxicChat. Протокол описан в `rq3_training/e5_guardrails/README.md`.

Перед запуском: **Runtime → Change runtime type → GPU**.
Результаты пишутся на Google Drive, чтобы не пропасть при отключении сессии.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Drive и код

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

OUT = '/content/drive/MyDrive/rq3_e5/outputs'            # полный эксперимент
OUT_SMOKE = '/content/drive/MyDrive/rq3_e5/outputs_smoke' # smoke-тест

In [ ]:
import os
REPO = '/content/Embedding-level-Guardrails'
# Приватный репозиторий: добавьте токен GITHUB_TOKEN в Colab Secrets (значок ключа слева).
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    url = f'https://{token}@github.com/embedding-level-guardrails/Embedding-level-Guardrails.git'
except Exception:
    url = 'https://github.com/embedding-level-guardrails/Embedding-level-Guardrails.git'

if not os.path.exists(REPO):
    !git clone -q -b rq3-anya {url} {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}/rq3_training/e5_guardrails
!git log -1 --format='%h %s'
os.environ['PYTHONPATH'] = 'src'

## 2. Подготовка данных AEGIS → ToxicChat

Ревизии датасетов и `split_seed` зафиксированы, поэтому сплиты совпадают с локальными:
train 19 766, validation 1 302, calibration 2 206, holdout 1 914, ToxicChat 4 951.

In [ ]:
!python -m e5_guardrails --set output_dir={OUT} prepare-data

## 3. Smoke-тест

200 строк на сплит, 3 шага, один seed. Проверяет, что все этапы отрабатывают;
цифры на такой выборке ничего не значат.

In [ ]:
!python -m e5_guardrails \
  --set output_dir={OUT_SMOKE} --set preprocess.max_rows_per_split=200 \
  --set training.epochs=1 --set training.steps_per_epoch=3 --set training.batch_size=8 \
  --set model.max_length=128 --set selection.probe_train_samples=100 --set selection.val_samples=100 \
  --set evaluation.bootstrap.n=50 \
  run-all --seeds 0

## 4. (Опционально) подбор гиперпараметров

Одинаковое число проб на каждый режим, выбор по source validation.
После этого полный прогон запускается с `--use-sweep`.

In [ ]:
# !python -m e5_guardrails --set output_dir={OUT} sweep --seed 0

## 5. Полный эксперимент: 3 seed × (frozen E5, CE, SupCon, CE + SupCon)

Сплиты уже подготовлены в шаге 2, поэтому `--skip-prepare`.

In [ ]:
!python -m e5_guardrails --set output_dir={OUT} run-all --seeds 0 1 2 --skip-prepare

## 6. Результаты

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

RES = f'{OUT}/aegis_to_toxicchat/results/main'
display(Markdown(open(f'{RES}/results.md').read()))

In [ ]:
per_seed = pd.read_csv(f'{RES}/results_per_seed.csv')
summary = pd.read_csv(f'{RES}/results_summary.csv')
boot = pd.read_csv(f'{RES}/bootstrap_ci.csv')

cols = ['model', 'test_set', 'n_seeds', 'fpr_mean', 'fnr_mean', 'tpr_at_fpr_mean', 'tpr_at_fpr_std', 'roc_auc_mean', 'avg_precision_mean']
summary.query("readout == 'linear_probe' and subset == 'all'")[cols].round(3)

In [ ]:
# Парные разности с frozen E5 (95% bootstrap CI по тестовым примерам, для каждого seed)
boot.query("minus == 'base' and metric == 'tpr_at_fpr'").round(3)